# 00 · Agent Foundations: how every agent in OutboundAI is built

**Start here.** The other three notebooks (post-call analysis, live suggestions, lead research) all reuse the
pattern you learn in this one.

**What you'll learn**

1. What "an agent" means in this project (hint: it is smaller and duller than you might think)
2. The `LLMProvider` abstraction: why agents never touch the Groq/OpenAI SDK directly
3. Prompts as data, and structured output with Pydantic
4. Retry-on-bad-JSON, and the two failure policies the repo uses
5. How to test an agent with **no API key** using a fake provider

**How to run:** `Runtime -> Run all`. With no API key the notebook runs in **FAKE mode** (canned outputs) so you can
follow the flow for free. To run against a real model, add `GROQ_API_KEY` (or `OPENAI_API_KEY`) in Colab's
left sidebar under the key icon (Secrets) and enable notebook access, then re-run.

## 1. The anatomy of an agent here

OutboundAI does **not** use one giant prompt. It uses several small, single-purpose agents. Each follows the same
five steps:

```
 inputs ──► build prompt ──► LLMProvider.complete_structured ──► validate vs Pydantic schema
                                      │                                   │
                                      │ bad JSON? retry once              ▼
                                      └───────────────────────►  typed result  OR  failure policy
```

| Piece | Repo location | Job |
|---|---|---|
| Provider | `app/ai/providers/` | One interface (`complete`, `complete_structured`); Groq/OpenAI hide behind it |
| Prompt | `app/ai/prompts/` | Templates live as data, so a prompt change is a small diff |
| Schema | `app/ai/schemas/` | Pydantic model describing the JSON the agent expects |
| Agent | `app/ai/agents/` | Glue: build prompt, call provider, apply failure policy |

**Why bother with this structure? (flexibility / scalability trade-offs)**

| You want to... | Cost with this structure |
|---|---|
| Swap Groq for OpenAI (or a new vendor) | Change the factory. Agents are untouched. **Cheap.** |
| Tweak wording of a prompt | Edit one prompt file. **Cheap.** |
| Test without spending money | Inject a fake provider. **Cheap.** |
| Run agents across many servers | Agents are stateless functions, so this part scales. State such as the live call store lives elsewhere. |
| Use a provider feature that is not in the interface (e.g. native tool calling quirks) | You must extend the interface, which touches every provider. **The price of the abstraction.** |

## 2. Setup

The next cell installs libraries. The one after it is the **shared setup**: a trimmed copy of the repo's
provider code. Read it once; it is the same in all four notebooks.

In [ ]:
!pip -q install pydantic openai

In [ ]:
# ---- Shared setup: identical in every notebook ---------------------------------------
# A trimmed copy of backend/app/ai/providers (base.py + factory.py). Agents only ever talk
# to an `LLMProvider`; they never know whether Groq, OpenAI or a fake is behind it.
import asyncio
import json
import os
import time
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Any, TypeVar

from pydantic import BaseModel, Field, ValidationError, field_validator

T = TypeVar("T", bound=BaseModel)


class LLMValidationError(RuntimeError):
    """complete_structured could not get valid JSON for the schema, even after one retry."""


@dataclass
class ProviderResponse:
    content: str | None


def _parse_json_content(content: str | None) -> Any:
    if not content:
        raise ValueError("empty response content")
    text = content.strip()
    if text.startswith("```"):  # models sometimes wrap JSON in a markdown fence
        text = text.strip("`").strip()
        if text.lower().startswith("json"):
            text = text[4:]
    return json.loads(text)


class LLMProvider(ABC):
    name: str

    @abstractmethod
    async def complete(self, messages, *, model, temperature=0.3, max_tokens=1000, response_format=None) -> ProviderResponse:
        """One chat-completion call."""

    async def complete_structured(self, messages, schema: type[T], *, model, temperature=0.3, max_tokens=1000) -> T:
        """Ask for JSON, validate it against `schema`, retry once with a fresh call on failure."""
        last_error: Exception | None = None
        for attempt in range(2):
            response = await self.complete(
                messages, model=model, temperature=temperature, max_tokens=max_tokens,
                response_format={"type": "json_object"},
            )
            try:
                return schema.model_validate(_parse_json_content(response.content))
            except (json.JSONDecodeError, ValidationError, ValueError, TypeError) as e:
                last_error = e
                print(f"  [retry] attempt {attempt + 1}/2 failed validation ({type(e).__name__})")
        raise LLMValidationError(f"{schema.__name__} validation failed after retry: {last_error}")


class OpenAICompatProvider(LLMProvider):
    """Real model. Groq exposes an OpenAI-compatible API, so one client class covers both here.
    (The repo has separate GroqProvider / OpenAIProvider classes; same idea.)"""

    def __init__(self, name: str, api_key: str, base_url: str | None = None):
        from openai import AsyncOpenAI

        self.name = name
        self._client = AsyncOpenAI(api_key=api_key, base_url=base_url)

    async def complete(self, messages, *, model, temperature=0.3, max_tokens=1000, response_format=None):
        kwargs = {"response_format": response_format} if response_format else {}
        start = time.monotonic()
        r = await self._client.chat.completions.create(
            model=model, messages=messages, temperature=temperature, max_tokens=max_tokens, **kwargs
        )
        print(f"  [LLM] provider={self.name} model={model} latency_ms={int((time.monotonic() - start) * 1000)}")
        return ProviderResponse(r.choices[0].message.content)


class FakeProvider(LLMProvider):
    """Offline stand-in: `responder(messages) -> str` returns canned/rule-based JSON text.
    This is how the repo's tests avoid real API calls (CODING_STANDARDS section 4)."""

    name = "fake"

    def __init__(self, responder):
        self.responder = responder
        self.calls: list[list[dict]] = []

    async def complete(self, messages, *, model, temperature=0.3, max_tokens=1000, response_format=None):
        self.calls.append(messages)
        return ProviderResponse(self.responder(messages))


GROQ_MODEL = "openai/gpt-oss-120b"  # settings.groq_model in the repo
GROQ_FAST_MODEL = "openai/gpt-oss-20b"  # settings.groq_fast_model (the small "gate" model)
OPENAI_MODEL = "gpt-4o"  # settings.openai_model


def _get_key(name: str) -> str | None:
    try:  # Colab: left sidebar -> key icon -> add a secret, enable notebook access
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name)


def make_provider(fake_responder):
    """Use a real provider when GROQ_API_KEY / OPENAI_API_KEY is available, else fall back to
    FakeProvider(fake_responder) so the whole notebook still runs offline."""
    if key := _get_key("GROQ_API_KEY"):
        llm, model, fast = OpenAICompatProvider("groq", key, "https://api.groq.com/openai/v1"), GROQ_MODEL, GROQ_FAST_MODEL
    elif key := _get_key("OPENAI_API_KEY"):
        llm, model, fast = OpenAICompatProvider("openai", key), OPENAI_MODEL, OPENAI_MODEL
    else:
        llm, model, fast = FakeProvider(fake_responder), "fake-model", "fake-model"
    if llm.name == "fake":
        print("MODE: FAKE (offline). Outputs are canned/rule-based, NOT generated by a model.")
        print("      Add GROQ_API_KEY or OPENAI_API_KEY as a Colab secret to run for real.")
    else:
        print(f"MODE: LIVE via {llm.name} (model={model})")
    return llm, model, fast

### Read the shared setup (30 seconds)

* `LLMProvider.complete()` is the **only** thing a provider must implement: messages in, text out.
* `complete_structured()` is written **once** on the base class: ask for JSON, validate, retry once. Every agent
  inherits it for free.
* `FakeProvider` takes a function `responder(messages) -> str`. That is the entire trick for offline testing.
* `make_provider(...)` picks a real provider if it finds a key, else the fake.

## 3. Step 1: talk to a provider

First, the smallest possible "agent": send messages, read the text back. Because we are in FAKE mode by default, we
supply a `responder`. In LIVE mode it is ignored.

In [ ]:
def demo_responder(messages):
    system = messages[0]["content"].lower()
    if "summary" in system:
        return json.dumps({"summary": "Rahul wants a case study; follow up Tuesday.", "sentiment": "Positive", "follow_up_needed": True})
    return json.dumps({"greeting": "hello from the fake provider"})


llm, MODEL, FAST_MODEL = make_provider(demo_responder)

reply = await llm.complete(
    [{"role": "user", "content": 'Reply with JSON: {"greeting": "..."}'}],
    model=MODEL,
    response_format={"type": "json_object"},
)
print("raw text from provider:", reply.content)

Note we used a top-level `await`. Colab (like Jupyter) already runs an event loop, so this works. In the FastAPI
backend the same functions are `await`ed inside request handlers and background tasks.

## 4. Step 2: prompts are data, schemas are contracts

Two small artifacts define the whole agent:

* a **prompt template** (a string with placeholders), exactly like `app/ai/prompts/analysis_prompts.py`
* a **Pydantic schema**, exactly like `app/ai/schemas/analysis.py`

In [ ]:
MINI_PROMPT = """You are an expert call analyst. Analyze this call transcript between a BA (Business Agent) and {client_name}.

Return a JSON object with exactly these fields:
{{
  "summary": "2-3 sentence summary, refer to the client by name ({client_name})",
  "sentiment": "Positive" or "Neutral" or "Negative",
  "follow_up_needed": true or false
}}"""


def build_mini_messages(transcript: str, contact_name: str) -> list[dict]:
    return [
        {"role": "system", "content": MINI_PROMPT.format(client_name=contact_name or "the client")},
        {"role": "user", "content": f"Call transcript:\n\n{transcript}\n\nAnalyze this call:"},
    ]


class MiniAnalysis(BaseModel):
    summary: str = ""
    sentiment: str = "Neutral"
    follow_up_needed: bool = False


messages = build_mini_messages("[You]: Hi Rahul...\n[Rahul]: Send me a case study.", "Rahul")
print(messages[0]["content"])

Notice the doubled braces `{{ }}` in the template: `str.format` treats `{x}` as a placeholder, so literal JSON
braces must be escaped. A very common first-time bug.

## 5. Step 3: structured output (the happy path)

`complete_structured` returns a **validated `MiniAnalysis` object**, not a dict or a string. Downstream code can rely
on the fields existing.

In [ ]:
result = await llm.complete_structured(messages, MiniAnalysis, model=MODEL, temperature=0.3, max_tokens=300)
print(type(result).__name__, "->", result)
print("sentiment:", result.sentiment, "| follow_up_needed:", result.follow_up_needed)

## 6. Step 4: models return bad JSON. Plan for it.

Real models occasionally return prose, truncated JSON, or the wrong types. Below, `FlakyProvider` returns garbage
on the first call and valid JSON on the second. Watch the `[retry]` line: the base class handled it, and the agent code
never knew.

In [ ]:
class FlakyProvider(LLMProvider):
    """Fails `bad_calls` times, then returns valid JSON. Used only to demonstrate retry."""

    name = "flaky"

    def __init__(self, bad_calls: int):
        self.bad_calls = bad_calls
        self.calls = 0

    async def complete(self, messages, *, model, temperature=0.3, max_tokens=1000, response_format=None):
        self.calls += 1
        if self.calls <= self.bad_calls:
            return ProviderResponse("Sure! Here is the analysis you asked for: it went well.")  # not JSON
        return ProviderResponse(json.dumps({"summary": "Recovered on retry.", "sentiment": "Positive"}))


one_bad = FlakyProvider(bad_calls=1)
ok = await one_bad.complete_structured(messages, MiniAnalysis, model="x")
print("result:", ok, "| provider was called", one_bad.calls, "times")

In [ ]:
always_bad = FlakyProvider(bad_calls=99)
try:
    await always_bad.complete_structured(messages, MiniAnalysis, model="x")
except LLMValidationError as e:
    print("raised LLMValidationError after", always_bad.calls, "calls:")
    print(" ", e)

Two attempts, then a typed exception. **What the agent does with that exception is a design decision**, and the repo
uses two different policies on purpose.

## 7. Step 5: failure policies (the most important design choice)

| Agent | On failure | Why |
|---|---|---|
| Post-call analysis | Returns `{"error": ..., "summary": "Analysis failed."}`, which is **stored** | The UI can show "analysis failed" and the history page can retry later |
| Live suggestion | Returns `None` / swallows everything | A coaching glitch must **never** break a live call. Silence is acceptable. |

Same exception, opposite handling. Run both against the always-bad provider:

In [ ]:
async def analyze_report_style(provider, transcript, name):
    """Post-call style: record the failure so it can be shown and retried."""
    try:
        result = await provider.complete_structured(build_mini_messages(transcript, name), MiniAnalysis, model="x")
        return result.model_dump()
    except Exception as e:
        return {"error": str(e), "summary": "Analysis failed."}


async def suggest_live_style(provider, transcript, name):
    """Live style: never raise, never block the call. Return None."""
    try:
        return await provider.complete_structured(build_mini_messages(transcript, name), MiniAnalysis, model="x")
    except Exception:
        return None


print("report style ->", await analyze_report_style(FlakyProvider(99), "...", "Rahul"))
print("live style   ->", await suggest_live_style(FlakyProvider(99), "...", "Rahul"))

> Rule of thumb you will reuse in the lead research agent: **if a human is waiting on a live conversation, fail
> silent; if the result is stored and reviewed later, fail loud and recorded.**

## 8. Step 6: tolerant schemas

LLMs are inconsistent: a field you asked for as a string may arrive as a dict, a list or `null`. The repo's
`Suggestion` schema absorbs this with `mode="before"` validators instead of failing and retrying. Compare a strict
schema with a tolerant one:

In [ ]:
class StrictSuggestion(BaseModel):
    recommended_project: str = ""


class TolerantSuggestion(BaseModel):
    recommended_project: str = ""

    @field_validator("recommended_project", mode="before")
    @classmethod
    def _flatten(cls, v):
        if isinstance(v, dict):  # tolerate {"name": ..., "pitch": ...}
            return " — ".join(str(x) for x in (v.get("name"), v.get("pitch")) if x)
        return str(v) if v else ""


messy = {"recommended_project": {"name": "ColdChain Track", "pitch": "cut dispatch errors by 30%"}}

try:
    StrictSuggestion.model_validate(messy)
except ValidationError as e:
    print("strict   -> ValidationError (would burn a retry):", e.errors()[0]["type"])

print("tolerant ->", TolerantSuggestion.model_validate(messy))

**Trade-off:** tolerance saves a ~1 s retry on the live path, but a validator that silently reshapes data can hide a
prompt that has drifted. Use it where latency matters (live agent) and keep schemas strict where you want to notice
problems (stored reports).

## 9. Step 7: test an agent without an API key

The repo's rule (`CODING_STANDARDS.md` section 4): agent tests use a fake `LLMProvider`. Here is a complete test of
a tiny agent. You can copy this shape into `backend/tests/ai/agents/`.

In [ ]:
async def tiny_agent(provider, transcript, name):
    result = await provider.complete_structured(build_mini_messages(transcript, name), MiniAnalysis, model="x")
    return result


fake = FakeProvider(lambda msgs: json.dumps({"summary": "ok", "sentiment": "Positive", "follow_up_needed": True}))
out = await tiny_agent(fake, "[Rahul]: send the case study", "Rahul")

assert isinstance(out, MiniAnalysis)
assert out.follow_up_needed is True
assert "Rahul" in fake.calls[0][0]["content"], "the client's name should be in the system prompt"
print("test passed ✅  | provider saw", len(fake.calls), "call(s)")

Because the fake records `calls`, you can also assert **what the agent sent**, not only what it returned. That
catches broken prompt-building without any model involved.

## 10. Exercises

1. **Add a field.** Add `key_points: list[str] = []` to `MiniAnalysis` and to the prompt. Does the fake still
   validate? (The schema has a default, so yes. What would happen if the field had *no* default?)
2. **Break the prompt on purpose.** Remove the doubled braces in `MINI_PROMPT` and read the error.
3. **Third failure policy.** Write `analyze_with_fallback` that retries on a *second* provider before giving up. Where
   would that logic live in the repo: the agent or the factory? (Think about what each choice costs.)
4. **Assert on the prompt.** Write a test that fails if `build_mini_messages("", "")` produces the phrase
   `the client` less than once.

## Where this lives in the repo

| Concept | File |
|---|---|
| Provider interface + retry | `backend/app/ai/providers/base.py` |
| Groq / OpenAI implementations | `backend/app/ai/providers/groq_provider.py`, `openai_provider.py` |
| Provider selection | `backend/app/ai/providers/factory.py` |
| Rules for all of this | `CODING_STANDARDS.md` section 2 |

**Next:** `01_post_call_analysis_agent.ipynb`